# Wisata Training: Decision-Driven Data Preparation dan Audit

Notebook ini menjadi jalur utama untuk data wisata MuterBandung. Polanya dibuat seperti notebook penginapan: setiap tahap kecil menjalankan pengecekan, melihat output, lalu mencatat keputusan singkat.

Catatan: notebook lama disimpan sebagai arsip. Tahap awal ini belum masuk ke LLM atau perubahan model.


## Tahap 0 - Menetapkan Environment dan Artefak

Tujuan: memastikan notebook membaca folder wisata yang benar dan memakai dataset curated terbaru sebagai basis audit.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 90)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Wisata_Workspace").exists():
            return candidate
    raise FileNotFoundError("Wisata_Workspace tidak ditemukan dari cwd saat ini.")

PROJECT_ROOT = find_project_root()
WISATA_DIR = PROJECT_ROOT / "Wisata_Workspace"
CURATED_DIR = WISATA_DIR / "01_Dataset" / "3_Curated"
DOC_DIR = WISATA_DIR / "03_Dokumentasi"

paths = {
    "dataset_utama": CURATED_DIR / "DATABASE_WISATA_LABELED_V2_REVIEWED.csv",
    "validation_json": CURATED_DIR / "data_validation_results.json",
    "validation_report": DOC_DIR / "DATA_VALIDATION_REPORT_2026-05-25.md",
    "sentiment_score": WISATA_DIR / "01_Dataset" / "SENTIMENT_SCORES_PER_LOKASI.csv",
    "manual_batch3": CURATED_DIR / "manual_review_batch3_remaining_46_after_status_facility_2026-05-27.csv",
}

artifact_status = []
for name, path in paths.items():
    artifact_status.append({
        "artifact": name,
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
        "modified_at": path.stat().st_mtime if path.exists() else None,
    })

artifact_status_df = pd.DataFrame(artifact_status)
display(artifact_status_df.drop(columns=["modified_at"]))
print("PROJECT_ROOT:", PROJECT_ROOT)


,artifact,relative_path,exists,size_kb
0,dataset_utama,Wisata_Workspace/01_Dataset/3_Curated/DATABASE_WISATA_LABELED_V2_REVIEWED.csv,True,502.21
1,validation_json,Wisata_Workspace/01_Dataset/3_Curated/data_validation_results.json,True,13.01
2,validation_report,Wisata_Workspace/03_Dokumentasi/DATA_VALIDATION_REPORT_2026-05-25.md,True,6.43
3,sentiment_score,Wisata_Workspace/01_Dataset/SENTIMENT_SCORES_PER_LOKASI.csv,True,27.45
4,manual_batch3,Wisata_Workspace/01_Dataset/3_Curated/manual_review_batch3_remaining_46_after_status_f...,True,21.12


PROJECT_ROOT: D:\File\file\Fauzan Lubada\PIJAK


### Keputusan: environment wisata layak dipakai

Semua artefak awal yang dibutuhkan tersedia. Untuk tahap ini, dataset utama yang dipakai adalah `DATABASE_WISATA_LABELED_V2_REVIEWED.csv`, bukan notebook lama atau file Colab eksperimen.


## Fase A - Lineage Dataset Utama

Tujuan: melihat jumlah data, jumlah kolom, ID unik, dan status snapshot dataset yang akan dipakai untuk audit berikutnya.


In [2]:
df = pd.read_csv(paths["dataset_utama"], dtype=str, keep_default_na=False)
validation = json.loads(paths["validation_json"].read_text(encoding="utf-8"))
summary = validation.get("summary", {})

lineage_summary = pd.DataFrame([
    {"metric": "rows_csv", "value": len(df)},
    {"metric": "columns_csv", "value": len(df.columns)},
    {"metric": "unique_location_id_csv", "value": df["location_id"].nunique() if "location_id" in df.columns else None},
    {"metric": "rows_validation_json", "value": summary.get("row_count")},
    {"metric": "columns_validation_json", "value": summary.get("column_count")},
    {"metric": "unique_location_id_validation", "value": summary.get("unique_location_id_count")},
    {"metric": "gate_status", "value": summary.get("gate_status")},
    {"metric": "validated_at", "value": summary.get("validated_at")},
])

display(lineage_summary)

id_check = pd.DataFrame([
    {"check": "location_id kosong", "count": int((df.get("location_id", pd.Series(dtype=str)) == "").sum())},
    {"check": "location_id duplikat", "count": int(df["location_id"].duplicated().sum()) if "location_id" in df.columns else None},
    {"check": "location_name kosong", "count": int((df.get("location_name", pd.Series(dtype=str)) == "").sum())},
])
display(id_check)


,metric,value
0,rows_csv,234
1,columns_csv,87
2,unique_location_id_csv,234
3,rows_validation_json,234
4,columns_validation_json,87
5,unique_location_id_validation,234
6,gate_status,PASS_WITH_WARNINGS
7,validated_at,2026-06-02T03:47:41.925336Z


,check,count
0,location_id kosong,0
1,location_id duplikat,0
2,location_name kosong,0


### Keputusan: snapshot wisata diterima sebagai basis audit

Jumlah baris, kolom, dan ID unik konsisten dengan hasil validation JSON. Tidak ada indikasi awal bahwa snapshot ini salah file, jadi tahap berikutnya boleh memakai dataset ini sebagai basis audit wisata.


## Fase B - Status Dataset dan Kolom Inti

Tujuan: memastikan status destinasi dan kolom penting untuk rekomendasi wisata tersedia sebelum masuk ke audit warning.


In [3]:
status_counts = df["display_status"].value_counts(dropna=False).rename_axis("display_status").reset_index(name="count") if "display_status" in df.columns else pd.DataFrame()
display(status_counts)

core_columns = [
    "location_id", "location_name", "category", "final_primary_intent", "final_core_labels",
    "display_status", "is_active_verified", "latitude", "longitude", "coordinate_verified",
    "price_min", "price_max", "price_type", "jam_buka_weekday", "jam_tutup_weekday",
    "avg_rating", "total_ulasan", "sentiment_score", "sentimen_label_lokasi",
    "sentiment_available", "media_available", "media_image_url", "parking_verified",
    "toilet_verified", "mushola_verified", "child_friendly_verified", "night_verified",
]

core_audit = []
for col in core_columns:
    exists = col in df.columns
    if exists:
        empty_count = int((df[col].astype(str).str.strip() == "").sum())
        non_empty_count = int(len(df) - empty_count)
    else:
        empty_count = None
        non_empty_count = None
    core_audit.append({
        "column": col,
        "exists": exists,
        "non_empty_count": non_empty_count,
        "empty_count": empty_count,
    })

core_audit_df = pd.DataFrame(core_audit)
display(core_audit_df)


,display_status,count
0,active_candidate,209
1,exclude_scope,19
2,temporarily_hidden,3
3,status_uncertain,3


,column,exists,non_empty_count,empty_count
0,location_id,True,234,0
1,location_name,True,234,0
2,category,True,234,0
3,final_primary_intent,True,234,0
4,final_core_labels,True,234,0
5,display_status,True,234,0
6,is_active_verified,True,87,147
7,latitude,True,234,0
8,longitude,True,234,0
9,coordinate_verified,True,234,0


### Keputusan: kolom inti cukup untuk audit rekomendasi

Kolom utama untuk status, koordinat, harga, jam buka, rating, sentiment, media, dan fasilitas sudah tersedia. Karena masih ada nilai kosong pada beberapa kolom, tahap berikutnya tetap perlu membaca warning validation, bukan langsung menganggap data final bersih.


## Fase C - Warning Validation yang Masih Aktif

Tujuan: membaca hasil validation otomatis dan menentukan bagian mana yang harus dibawa ke audit lanjutan.


In [4]:
issues = pd.DataFrame(validation.get("issues", []))
if not issues.empty:
    issue_cols = ["severity", "code", "field", "count", "message"]
    issues_view = issues[issue_cols].sort_values(["severity", "count"], ascending=[True, False]).reset_index(drop=True)
    display(issues_view)
else:
    issues_view = pd.DataFrame(columns=["severity", "code", "field", "count", "message"])
    display(issues_view)

sample_rows = []
for issue in validation.get("issues", []):
    for row in issue.get("sample_rows", [])[:5]:
        sample_rows.append({
            "code": issue.get("code"),
            "field": issue.get("field"),
            "location_id": row.get("location_id"),
            "location_name": row.get("location_name"),
        })

sample_df = pd.DataFrame(sample_rows)
display(sample_df.head(30))

summary_view = pd.DataFrame([
    {"metric": "gate_status", "value": summary.get("gate_status")},
    {"metric": "error_count", "value": summary.get("error_count")},
    {"metric": "warning_count", "value": summary.get("warning_count")},
    {"metric": "info_count", "value": summary.get("info_count")},
    {"metric": "active_candidate_count", "value": summary.get("active_candidate_count")},
])
display(summary_view)


,severity,code,field,count,message
0,INFO,I_ACTIVE_WEBSITE_MISSING,media_website,139,Active candidates without official/reference website.
1,WARNING,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,147,Active candidates missing is_active_verified must not be described as verified.
2,WARNING,W_ACTIVE_FACILITY_VERIFICATION_SPARSE,facility_verified_flags,146,Active candidates with no verified facility flags require conservative LLM wording.
3,WARNING,W_ACTIVE_MEDIA_UNAVAILABLE,media_available,39,Active candidates without media need conservative UI/LLM presentation.
4,WARNING,W_ACTIVE_RATING_MISSING,avg_rating,16,Active candidates with missing avg_rating rely on runtime median fallback.
5,WARNING,W_ACTIVE_SENTIMENT_UNAVAILABLE,sentiment_available,16,Active candidates with unavailable sentiment need neutral/caveated LLM wording.
6,WARNING,W_ACTIVE_COORDINATE_UNVERIFIED,coordinate_verified,9,Active candidates with coordinate_verified=False should be treated carefully for dista...
7,WARNING,W_ZERO_PRICE_NOT_MARKED_FREE,"price_type,price_max",1,Rows with zero max price should usually be marked Gratis.
8,WARNING,W_OPEN_24H_FLAG_HOURS_MISMATCH,open_24h_verified,1,open_24h_verified=True should align with 00:00-23:59 weekday/weekend hours.


,code,field,location_id,location_name
0,W_ACTIVE_RATING_MISSING,avg_rating,LOC-029,Dusun Bambu
1,W_ACTIVE_RATING_MISSING,avg_rating,LOC-068,Pasar Cimindi
2,W_ACTIVE_RATING_MISSING,avg_rating,LOC-133,Punclut Bandung (Puncak Ciumbuleuit)
3,W_ACTIVE_RATING_MISSING,avg_rating,LOC-134,Bukit Bintang Bandung (Patahan Lembang)
4,W_ACTIVE_RATING_MISSING,avg_rating,LOC-152,Kebun Teh Rancabali
5,W_ZERO_PRICE_NOT_MARKED_FREE,"price_type,price_max",LOC-233,Kampung Wisata Pangjugjugan
6,W_OPEN_24H_FLAG_HOURS_MISMATCH,open_24h_verified,LOC-134,Bukit Bintang Bandung (Patahan Lembang)
7,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,LOC-001,23 Paskal Shopping Center
8,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,LOC-002,Alam Wisata Cimahi
9,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,LOC-003,Alun-Alun Kota Bandung


,metric,value
0,gate_status,PASS_WITH_WARNINGS
1,error_count,0
2,warning_count,375
3,info_count,139
4,active_candidate_count,209


### Keputusan: dataset bisa lanjut, tetapi masih perlu caveat

Gate validation berada di `PASS_WITH_WARNINGS`. Tidak ada error aktif, tetapi warning fasilitas, status aktif, media, rating, sentiment, dan koordinat perlu dibawa ke tahap audit berikutnya. Untuk LLM dan rekomendasi, sistem belum boleh membuat klaim yang terlalu pasti pada field yang belum verified.


## Ringkasan Tahap Awal

| Area | Keputusan |
|---|---|
| Dataset utama | Dipakai: `DATABASE_WISATA_LABELED_V2_REVIEWED.csv` |
| Notebook lama | Disimpan sebagai arsip, bukan jalur utama |
| Status validasi | `PASS_WITH_WARNINGS` |
| Lanjut ke LLM | Belum, audit data wisata dilanjutkan dulu |
| Tahap berikutnya | Audit targeted completion, sentiment, dan kesiapan filter website |

Tahap ini sengaja dibatasi agar perubahan notebook aman dan mudah dicek dulu sebelum lanjut ke fase berikutnya.
